In [1]:
# Initialize Otter
import otter
grader = otter.Notebook("lab.ipynb")

c:\Users\jwang\Miniforge3\envs\dsc80\Lib\site-packages\otter\export\__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


# Lab 6 – APIs and Web Scraping

## DSC 80, Fall 2025

### Due Date: Monday, November 10th at 11:59 PM


## Instructions

Welcome to the sixth DSC 80 lab this quarter!

Much like in DSC 10, this Jupyter Notebook contains the statements of the problems and provides code and Markdown cells to display your answers to the problems. Unlike DSC 10, the notebook is *only* for displaying a readable version of your final answers. The coding will be done in an accompanying `lab.py` file that is imported into the current notebook, and **you will only submit that `lab.py` file**, not this notebook!

Some additional guidelines:
- **Unlike in DSC 10, labs will have both public tests and hidden tests.** The bulk of your grade will come from your scores on hidden tests, which you will only see on Gradescope after the assignment deadline.
- **Do not change the function names in the `lab.py` file!** The functions in the `lab.py` file are how your assignment is graded, and they are graded by their name. If you changed something you weren't supposed to, you can find the original code in the [course GitHub repository](https://github.com/dsc-courses/dsc80-2025-sp).
- Notebooks are nice for testing and experimenting with different implementations before designing your function in your `lab.py` file. You can write code here, but make sure that all of your real work is in the `lab.py` file, since that's all you're submitting.
- You are encouraged to write your own additional helper functions to solve the lab, as long as they also end up in `lab.py`.

**To ensure that all of the work you want to submit is in `lab.py`, we've included a script named `lab-validation.py` in the lab folder. You shouldn't edit it, but instead, you should call it from the command line (e.g. the Terminal) to test your work.** More details on its usage are given at the bottom of this notebook.

**Importing code from `lab.py`**:

* Below, we import the `.py` file that's contained in the same directory as this notebook.
* We use the `autoreload` notebook extension to make changes to our `lab.py` file immediately available in our notebook. Without this extension, we would need to restart the notebook kernel to see any changes to `lab.py` in the notebook.
    - `autoreload` is necessary because, upon import, `lab.py` is compiled to bytecode (in the directory `__pycache__`). Subsequent imports of `lab` merely import the existing compiled python.

<div class="alert alert-block alert-danger">
<b>Note: </b> For this lab, due to system constraints, the code may take varying amounts of time to run on different machines. If you are able to submit to Gradescope and the code runs, then you have met the runtime requirements.
</div>

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from lab import *

If the cell below returns a `ModuleNotFoundError`, please run `!pip install lxml` in a new cell. After `lxml` is succesfully installed, go to `kernel` then restart. Note that you will only need to run `!pip install lxml` once. 

In [4]:
import os
import pandas as pd
import numpy as np
import requests
import bs4
import lxml

In [5]:
# !pip install lxml

## Question 1 – Practice with HTML Tags 📎

In Question 2, you'll spend plenty of time parsing HTML source code. But before you get your hands dirty trying to extract information from HTML written by other people, it is a good idea to write basic HTML code yourself. This exercise will help you better understand how the code in a `.html` file is structured.

For this question, you'll create a very basic `.html` file, named `lab06_1.html`, that satisfies the following conditions:

- It must have `<title>` and `<head>` tags.
- It must also have `<body>` tags. Within the `<body>` tags, it must have:
    - At least two headers.
    * At least three images.
        - At least one image must be a local file.
        - At least one image must be linked to online source.
        - At least one image has to have default text when it cannot be displayed.
    * At least three references (hyperlinks) to different web pages.
    * At least one table with two rows and two columns.
    

Make sure to save your file as `lab06_1.html`, and save it in the same directory as `lab.py`. **When submitting this homework to Gradescope, make sure to also upload `lab06_1.html` along with the local image that you embedded in your site.** You can upload multiple files to Gradescope at a time.
   

***Notes***:
- You can write and view basic HTML with a Jupyter Notebook, using either a Markdown cell or by using the `IPython.display.HTML` function (which takes in a string of HTML and renders it).
- If you write your HTML code within a Jupyter Notebook, you should later copy your code into a text editor and save it with the `.html` extension. You could also write your HTML in a text editor directly.
- Be sure to open your final `.html` file in a browser and make sure it looks correct on its own.

In [6]:
# Don't delete this cell!
question1()

In [7]:
grader.check("q1")

q1 results: All test cases passed!

## Question 2 – Scraping an Online Bookstore 📚

Browse through the following fake online bookstore: http://books.toscrape.com/. This website is meant for toying with scraping.

Your job is to scrape the website, collecting data on all books that have:
- **_at least_ a four-star rating**, and
- **a price _strictly_ less than £50**, and 
- **belong to specific categories** (more details below). 

You will extract the information into a DataFrame that looks like the one below.

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>UPC</th>
      <th>Product Type</th>
      <th>Price (excl. tax)</th>
      <th>Price (incl. tax)</th>
      <th>Tax</th>
      <th>Availability</th>
      <th>Number of reviews</th>
      <th>Category</th>
      <th>Rating</th>
      <th>Description</th>
      <th>Title</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>e10e1e165dc8be4a</td>
      <td>Books</td>
      <td>Â£22.60</td>
      <td>Â£22.60</td>
      <td>Â£0.00</td>
      <td>In stock (19 available)</td>
      <td>0</td>
      <td>Default</td>
      <td>Four</td>
      <td>For readers of Laura Hillenbrand's Seabiscuit...</td>
      <td>The Boys in the Boat: Nine Americans...</td>
    </tr>
    <tr>
      <th>1</th>
      <td>c2e46a2ee3b4a322</td>
      <td>Books</td>
      <td>Â£25.27</td>
      <td>Â£25.27</td>
      <td>Â£0.00</td>
      <td>In stock (19 available)</td>
      <td>0</td>
      <td>Romance</td>
      <td>Five</td>
      <td>A Michelin two-star chef at twenty-eight, Violette...</td>
      <td>Chase Me (Paris Nights #2)</td>
    </tr>
    <tr>
      <th>2</th>
      <td>00bfed9e18bb36f3</td>
      <td>Books</td>
      <td>Â£34.53</td>
      <td>Â£34.53</td>
      <td>Â£0.00</td>
      <td>In stock (19 available)</td>
      <td>0</td>
      <td>Romance</td>
      <td>Five</td>
      <td>No matter how busy he keeps himself...</td>
      <td>Black Dust</td>
    </tr>
  </tbody>
</table>

To do so, implement the following functions.

<br>

#### `extract_book_links`

Complete the implementation of the function `extract_book_links`, which takes in the content of a page that contains book listings as a **string of HTML**, and returns a **list** of URLs of book-specific pages for all books with **_at least_ a four-star rating and a price _strictly_ less than £50**.

For this method, the URLs you return should not contain the protocol (i.e. `'https://'`). The protocols should be added back into the URLs when you actually make the requests.


<br>

#### `get_product_info`

Complete the implementation of the function `get_product_info`, which takes in the content of a book-specific page as a **string of HTML**, and a list `categories` of book categories. If the input book is in the list of `categories`, `get_product_info` should return a dictionary corresponding to a row in the DataFrame in the image above (where the keys are the column names and the values are the row values). If the input book is not in the list of `categories`, return `None`. <b> The order of the dictionary keys does not matter. </b>


<br>

#### `scrape_books`

Finally, put everything together. Complete the implementation of the function `scrape_books`, which takes in an integer `k` and a list `categories` of book categories. `scrape_books` should use `requests` to scrape the first `k` pages of the bookstore and return a DataFrame of only the books that have:
- **_at least_ a four-star rating**, and
- **a price _strictly_ less than £50**, and
- **a category that is in the list `categories`**.

<b> The order of the DataFrame columns does not matter. </b>

<br>

Some general guidance and tips:

- The first page of the bookstore is at http://books.toscrape.com/catalogue/page-1.html. Subsequent pages can be found by clicking the "Next" button at the bottom of the page. Look at how the URLs change each time you navigate to a new page; think about how to use f-strings (or some other string formatting technique) to generate these URLs.
- Use "inspect element" to view the source code of the pages you're trying to scrape. To find a book's category, look at the hyperlinks in the book-specific page for that book.
- **`scrape_books` should run in under 180 seconds on the entire bookstore (`k = 50`). `scrape_books` is also the only function that should make `GET` requests; the other two functions parse already-existing HTML.**
- When instantiating `bs4.BeautifulSoup` objects, use the optional argument `features='lxml'` to suppress any warnings.
- Don't worry about typecasting, i.e. it's fine if `'Number of reviews'` is not stored as type `int`. Also, don't worry if you run into encoding errors in your price columns (as the example DataFrame at the top of this cell contains).

In [8]:
def extract_book_links(text):
    soup = bs4.BeautifulSoup(text, 'lxml')
    books = soup.find_all("li", attrs={"class": "col-xs-6 col-sm-4 col-md-3 col-lg-3"})
    prices = [(float)(books[i].find("p", attrs={"class": "price_color"}).text.replace("£", "").replace("Â", "")) for i in range(len(books))]
    text_to_nums = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    star_ratings = [(text_to_nums.get(books[i].find("p", attrs={"class": "star-rating"}).get("class")[1])) for i in range(len(books))]
    titles = [books[i].find("h3").find("a").get("title") for i in range(len(books))]
    links = [books[i].find("h3").find("a").get("href") for i in range(len(books))]
    df = pd.DataFrame({"star ratings": star_ratings, "prices": prices, "href": links}, index=titles)
    return list(df[(df["star ratings"] >= 4) & (df["prices"] <50)]["href"].values)

In [9]:
extract_book_links_fp = os.path.join('data', 'products.html')
extract_book_out = extract_book_links(
    open(extract_book_links_fp, encoding='utf-8').read()
)
extract_book_out

['seven-brief-lessons-on-physics_219/index.html',
 'scarlet-the-lunar-chronicles-2_218/index.html',
 'saga-volume-3-saga-collected-editions-3_216/index.html',
 'running-with-scissors_215/index.html',
 'rise-of-the-rocket-girls-the-women-who-propelled-us-from-missiles-to-the-moon-to-mars_213/index.html',
 'ready-player-one_209/index.html']

In [10]:
def get_product_info(text, categories):
    soup = bs4.BeautifulSoup(text, 'lxml')
    tables = soup.find("table")
    lst = []
    table_tr = tables.find_all("tr")
    for row in table_tr:
        lst.append(row.find("td").text)
    category = soup.find("ul", attrs={"class": "breadcrumb"}).find_all("a")[-1].text
    star_rating = soup.find("div", attrs={"class": "col-sm-6 product_main"}).find("p", attrs={"class": "star-rating"}).get("class")[1]
    description = soup.find("meta", attrs={"name": "description"}).get("content").strip()
    title = soup.find("div", attrs={"class": "col-sm-6 product_main"}).find("h1").text
    lst.extend([category, star_rating, description, title])
    col_names = ["UPC", "Product Type", "Price (excl. tax)", "Price (incl. tax)", "Tax", "Availability", "Number of reviews", "Category", "Rating", "Description", "Title"]
    dct = dict(zip(col_names, lst))
    if dct.get("Category") in categories:
        return dct
    else:
        return None

In [11]:
extract_book_url = 'scarlet-the-lunar-chronicles-2_218/index.html'

# doc tests for get product info
get_product_info_fp = os.path.join('data', 'Frankenstein.html')
get_product_info_out = get_product_info(
    open(get_product_info_fp, encoding='utf-8').read(), ['Default']
)

get_product_info_out

{'UPC': 'a492f49a3e2b6a71',
 'Product Type': 'Books',
 'Price (excl. tax)': '£38.00',
 'Price (incl. tax)': '£38.00',
 'Tax': '£0.00',
 'Availability': 'In stock (1 available)',
 'Number of reviews': '0',
 'Category': 'Default',
 'Rating': 'Two',
 'Description': "Mary Shelley began writing Frankenstein when she was only eighteen. At once a Gothic thriller, a passionate romance, and a cautionary tale about the dangers of science, Frankenstein tells the story of committed science student Victor Frankenstein. Obsessed with discovering the cause of generation and life and bestowing animation upon lifeless matter, Frankenstein assembles Mary Shelley began writing Frankenstein when she was only eighteen. At once a Gothic thriller, a passionate romance, and a cautionary tale about the dangers of science, Frankenstein tells the story of committed science student Victor Frankenstein. Obsessed with discovering the cause of generation and life and bestowing animation upon lifeless matter, Franken

In [12]:
def scrape_books(k, categories):
    pages = []
    for i in range(1, k + 1):
        response = requests.get(f"https://books.toscrape.com/catalogue/page-{i}.html")
        pages.append(response.text)
    links = []
    for page in pages:
        links.extend(extract_book_links(page))
    rows = []
    for link in links:
        response2 = requests.get(f"https://books.toscrape.com/catalogue/{link}").text
        rows.append(get_product_info(response2, categories))
    valid_rows = []
    for row in rows:
        if row != None:
            valid_rows.append(row)
    return pd.DataFrame(valid_rows)

In [13]:
scrape_books_out = scrape_books(50, ['Default', 'Romance'])
scrape_books_out

,UPC,Product Type,Price (excl. tax),Price (incl. tax),Tax,Availability,Number of reviews,Category,Rating,Description,Title
0,e10e1e165dc8be4a,Books,Â£22.60,Â£22.60,Â£0.00,In stock (19 available),0,Default,Four,For readers of Laura Hillenbrand's Seabiscuit ...,The Boys in the Boat: Nine Americans and Their...
1,c2e46a2ee3b4a322,Books,Â£25.27,Â£25.27,Â£0.00,In stock (19 available),0,Romance,Five,"A Michelin two-star chef at twenty-eight, Viol...",Chase Me (Paris Nights #2)
2,00bfed9e18bb36f3,Books,Â£34.53,Â£34.53,Â£0.00,In stock (19 available),0,Romance,Five,"No matter how busy he keeps himself, successfu...",Black Dust
3,8c9e6bf2467d740d,Books,Â£20.59,Â£20.59,Â£0.00,In stock (16 available),0,Default,Five,"Slay Procrastination, Distraction, and Overwhe...",The Inefficiency Assassin: Time Management Tac...
4,9e28048cea8d41f7,Books,Â£15.97,Â£15.97,Â£0.00,In stock (16 available),0,Romance,Four,"Caleb Stone was raised on the Upper East Side,...",First and First (Five Boroughs #3)
5,961f18db4f138211,Books,Â£27.37,Â£27.37,Â£0.00,In stock (15 available),0,Default,Four,Fragments of a Great Secret have been found in...,The Secret (The Secret #1)
6,eac1a180047ad54e,Books,Â£41.82,Â£41.82,Â£0.00,In stock (15 available),0,Default,Four,Khaled Hosseini's #1 New York Times Bestsellin...,The Kite Runner
7,e7469e22b5bfb3e7,Books,Â£33.34,Â£33.34,Â£0.00,In stock (15 available),0,Default,Five,"Conflict is an inevitable part of life, accord...",The Art of War
8,ca71e72655bece85,Books,Â£16.24,Â£16.24,Â£0.00,In stock (15 available),0,Romance,Four,Katy Lewis has it all: a sports reporting job ...,Something More Than This
9,6ffb36aaeff1c81e,Books,Â£34.95,Â£34.95,Â£0.00,In stock (15 available),0,Default,Four,Forget everything you thought you knew about h...,Drive: The Surprising Truth About What Motivat...


In [14]:
# don't delete this cell, but do run it -- it is needed for the autograder tests

# public test for extract_book_links 
extract_book_links_fp = os.path.join('data', 'products.html')
extract_book_out = extract_book_links(
    open(extract_book_links_fp, encoding='utf-8').read()
)
extract_book_url = 'scarlet-the-lunar-chronicles-2_218/index.html'

# doc tests for get product info
get_product_info_fp = os.path.join('data', 'Frankenstein.html')
get_product_info_out = get_product_info(
    open(get_product_info_fp, encoding='utf-8').read(), ['Default']
)

# public test for scrape books 
scrape_books_out = scrape_books(1, ['Mystery'])

In [15]:
grader.check("q2")

q2 results: All test cases passed!

## Question 3 – API Requests 🤑

<div class="alert alert-block alert-danger">
<b>Note: </b> Unfortunately, the API that we used in this problem is no longer free. Because of this, we've removed Question 3, and you can simply move on to Question 4.
</div>


## Question 4 – Comment Threads 🧵

You regularly browse [Hacker News](https://news.ycombinator.com/) to keep up with the latest news in tech. One link to a Hacker News article is https://news.ycombinator.com/item?id=18344932. Note that this article has 18 comments and has a `storyid` of 18344932.

The problem now is that you don't have internet access on your phone during your morning commute to work, so you want to save the interesting stories' comment threads beforehand in a CSV. You find their [API documentation](https://github.com/HackerNews/API) and decide to get to work.

Complete the implementation of the function `get_comments`, which takes in a `storyid` and returns a DataFrame of all the comments below the news story. You can ignore "dead" comments(you will know them when you see them), as well as "dead" comments’ children. **Make sure the order of the comments in your DataFrame is from top to bottom just as you see on the website**. 

The DataFrame that `get_comments` returns should have 5 columns:
1. `'id'`: The unique ID of the comment.
2. `'by'`: The author of the comment.
3. `'text'`: The actual comment.
4. `'parent'`: The unique ID of the comment this comment is replying to.
5. `'time'`: When the comment was created (in `pd.Timestamp` format).

Some guidance:
1. The URL to make requests to is `'https://hacker-news.firebaseio.com/v0/item/{}.json'`, however, the `{}` should be replaced with the ID of the article or page you are trying to access. 
2. Again, do not `import json` – instead, use the `json` method on the Response object you get back.
3. Use depth-first search when traversing the comments tree. You will have to do this manually, since you cannot use Beautiful Soup (which is only for HTML documents, not JSON objects).
4. Make sure the length of your returned DataFrame is the same as value for the `'descendants'` key in the response JSON (both of which correspond to the number of comments for the story).
5. You are allowed to use loops in this function. You may also want to create at least one helper function.

<div class="alert alert-block alert-success">
    You may find <a href="https://www.youtube.com/watch?v=uOfwW-onmpc"><b>this hint video 🎥</b></a> helpful!
</div>

In [16]:
def get_comments1(storyid):
    response = requests.get(f"https://news.ycombinator.com/item?id={storyid}").text
    soup = bs4.BeautifulSoup(response, "lxml")
    comments = soup.find_all("tr", attrs={"class": "athing comtr"})
    ids = [int(comments[i].get("id")) for i in range(len(comments))]
    names = [comments[i].find("a", attrs={"class": "hnuser"}).text for i in range(len(comments))]
    comments_txt = [comments[i].find("div", attrs={"class": "commtext c00"}).text for i in (range(len(comments)))]
    parent = [int(comments[i].find("span", attrs={"class": "navs"}).find("a").get("href").strip("#")) for i in range(len(comments))]
    time = [pd.Timestamp(comments[i].find("span", attrs={"class", "age"}).get("title").split()[0]) for i in range(len(comments))]
    columns = ["id", "by", "text", "parent", "time"]
    return pd.DataFrame([ids, names, comments_txt, parent, time], index=columns).T

In [17]:
out_expected = get_comments1(18344932)
out_expected

,id,by,text,parent,time
0,18380397,valyala,TimescaleDB is great for storing time series c...,18346406,2018-11-05 06:53:19
1,18346406,msiggy,I'm excited to give this database a try if I c...,18380397,2018-10-31 15:20:22
2,18348601,sman393,Can this be used side by side on normal Postgr...,18346406,2018-10-31 19:29:39
3,18348631,RobAtticus,"Yep, absolutely. Regular PostgreSQL tables coe...",18348601,2018-10-31 19:34:52
4,18348984,sman393,Good to hear! how does the current TimescaleDB...,18348601,2018-10-31 20:23:46
5,18349540,RobAtticus,Not sure I follow exactly what you're asking. ...,18348601,2018-10-31 21:47:20
6,18350673,sman393,Alright thanks! I thought I read that Timescal...,18348601,2018-11-01 01:11:59
7,18351061,RobAtticus,It does not support sharding writes across mul...,18348601,2018-11-01 02:35:03
8,18346750,zip1234,How fast is it when it has a TB of data? I rea...,18348601,2018-10-31 15:51:43
9,18347260,nevi-me,I spent about 8 months writing data to TSDB. I...,18346750,2018-10-31 16:47:34


In [18]:
# def get_comments(storyid):

#     def get_current_info(id):
#         current_id = id
#         response = requests.get(f"https://hacker-news.firebaseio.com/v0/item/{id}.json").text
#         by_start = response.find('"by":"') + len('"by":"')
#         by_stop = response.find('",', by_start)
#         by = response[by_start:by_stop]
#         id_start = response.find('"id":') + len('"by":')
#         id_stop = response.find(',', id_start)
#         id = int(response[id_start:id_stop])
#         text_start = response.find('"text":"') + len('"text":"')
#         text_stop = response.find(',"time":', text_start)
#         text = response[text_start:text_stop]
#         parent_start = response.find('"parent":') + len('"parent":')
#         parent_stop = response.find(',', parent_start)
#         parent = int(response[parent_start:parent_stop])
#         time_start = response.find('"time":') + len('"time":')
#         time_stop = response.find(',"type":', time_start)
#         time = pd.to_datetime(int(response[time_start:time_stop]), unit='s')
#         if ("kids" in response):
#             kids_start = response.find('"kids":[') + len('"kids":[')
#             kids_stop = response.find('"kids":[', kids_start)
#             kids = response[kids_start:kids_stop].split(",")
#             return get_current_info(kids[0])
#         return [current_id, by, text, parent, time]

#     response = requests.get(f"https://hacker-news.firebaseio.com/v0/item/{storyid}.json").text
#     start = response.find('"kids":') + len('"kids":[')
#     stop = response.find(']', start)
#     ids = response[start:stop].split(",")
#     for i in range(len(ids)):
#         ids[i] = int(ids[i])
#     info = []
#     for id in ids:
#         info.append(get_current_info(id))
#     id_lst = []
#     user_lst = []
#     text_lst = []
#     parent_lst = []
#     time_lst = []
#     for i in info:
#         id_lst.append(i[0])
#         user_lst.append(i[1])
#         text_lst.append(i[2])
#         parent_lst.append(i[3])
#         time_lst.append(i[4])
#     columns = ["id", "by", "text", "parent", "time"]
#     return pd.DataFrame([id_lst, user_lst, text_lst, parent_lst, time_lst], index=columns).T


In [19]:
# def get_comments(storyid):

#     def get_current_info(id):
#         response = requests.get(f"https://hacker-news.firebaseio.com/v0/item/{id}.json")
#         data = response
#         if not data or data.get("dead") or data.get("deleted"):
#             return []

#         current_id = data.get("id")
#         by = data.get("by")
#         text = data.get("text")
#         parent = data.get("parent")
#         time = pd.to_datetime(data.get("time"), unit='s')

#         info = [[current_id, by, text, parent, time]]

#         if "kids" in data:
#             for kid in data["kids"]:
#                 info.extend(get_current_info(kid))

#         return info

#     response = requests.get(f"https://hacker-news.firebaseio.com/v0/item/{storyid}.json")
#     story_data = response.json()

#     info = []
#     if "kids" in story_data:
#         for id in story_data["kids"]:
#             info.extend(get_current_info(id))

#     columns = ["id", "by", "text", "parent", "time"]
#     df = pd.DataFrame(info, columns=columns)

    # return df

In [20]:
def get_comments(storyid):

    def get_current_info(id):
        response = requests.get(f"https://hacker-news.firebaseio.com/v0/item/{id}.json").json()
        if response.get("dead") or response.get("deleted"):
            return []

        current_id = response.get("id")
        by = response.get("by")
        text = response.get("text")
        parent = response.get("parent")
        time = pd.to_datetime(response.get("time"), unit='s')

        temp = [[current_id, by, text, parent, time]]

        kids = response.get("kids")
        if response.get("kids") != None:
            for kid in kids:
                temp.extend(get_current_info(kid))
            return temp
        else:
            return temp
        
    data = get_current_info(storyid)
    cols = ["id", "by", "text", "parent", "time"]
    return pd.DataFrame(data[1:], columns=cols)
        
out_expected["id"][0]

18380397

In [21]:
comments = get_comments(18344932)
comments

,id,by,text,parent,time
0,18380397,valyala,TimescaleDB is great for storing time series c...,18344932,2018-11-05 06:53:19
1,18346406,msiggy,I&#x27;m excited to give this database a try i...,18344932,2018-10-31 15:20:22
2,18348601,sman393,Can this be used side by side on normal Postgr...,18344932,2018-10-31 19:29:39
3,18348631,RobAtticus,"Yep, absolutely. Regular PostgreSQL tables coe...",18348601,2018-10-31 19:34:52
4,18348984,sman393,Good to hear! how does the current TimescaleDB...,18348631,2018-10-31 20:23:46
5,18349540,RobAtticus,Not sure I follow exactly what you&#x27;re ask...,18348984,2018-10-31 21:47:20
6,18350673,sman393,Alright thanks! I thought I read that Timescal...,18349540,2018-11-01 01:11:59
7,18351061,RobAtticus,It does not support sharding writes across mul...,18350673,2018-11-01 02:35:03
8,18346750,zip1234,How fast is it when it has a TB of data? I rea...,18344932,2018-10-31 15:51:43
9,18347260,nevi-me,I spent about 8 months writing data to TSDB. I...,18346750,2018-10-31 16:47:34


In [22]:
out_expected == comments

,id,by,text,parent,time
0,True,True,False,False,True
1,True,True,False,False,True
2,True,True,True,False,True
3,True,True,False,True,True
4,True,True,True,False,True
5,True,True,False,False,True
6,True,True,False,False,True
7,True,True,True,False,True
8,True,True,True,False,True
9,True,True,False,True,True


In [23]:
# don't delete this cell, but do run it -- it is needed for the autograder tests
comments = get_comments(18344932)

In [24]:
grader.check("q3")

q3 results: All test cases passed!

## Congratulations! You're done Lab 6! 🏁

As a reminder, all of the work you want to submit needs to be in `lab.py`.

To ensure that all of the work you want to submit is in `lab.py`, we've included a script named `lab-validation.py` in the lab folder. You shouldn't edit it, but instead, you should call it from the command line (e.g. the Terminal) to test your work.

Once you've finished the lab, you should open the command line and run, in the directory for this lab:

```
python lab-validation.py
```

**This will run all of the `grader.check` cells that you see in this notebook, but only using the code in `lab.py` – that is, it doesn't look at any of the code in this notebook. If all of your `grader.check` cells pass in this notebook but not all of them pass in your command line with the above command, then you likely have code in your notebook that isn't in your `lab.py`!**

You can also use `lab-validation.py` to test individual questions. For instance,

```
python lab-validation.py q1 q2 q4
```

will run the `grader.check` cells for Questions 1, 2, and 4 – again, only using the code in `lab.py`. [This video](https://www.loom.com/share/0ea254b85b2745e59322b5e5a8692e91?sid=5acc92e6-0dfe-4555-9b6a-8115b6a52f99) how to use the script as well.

Once `python lab-validation.py` shows that you're passing all test cases, you're ready to submit your `lab.py` (and only your `lab.py`) to Gradescope. Once submitting to Gradescope, make sure to stick around until all test cases pass.

There is also a call to `grader.check_all()` below in _this_ notebook, but make sure to also follow the steps above.

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [25]:
grader.check_all()

q1 results: All test cases passed!

q2 results: All test cases passed!

q3 results: All test cases passed!